# MicroWakeWord V2 Trainer

This notebook trains a custom wake word model using:
- **Piper TTS** for synthetic training sample generation
- **Wyoming protocol** for Home Assistant voice stack integration
- **TensorFlow** for model training with INT8 quantization
- **ESPHome / ESP32** deployment target

### Prerequisites
Install dependencies:
```bash
pip install -r requirements.txt
```

### Workflow Overview
1. Configure wake word and parameters
2. Generate synthetic samples via Piper TTS
3. Augment audio with noise/RIR
4. Extract mel-spectrogram features
5. Train model (openWakeWord + microWakeWord architectures)
6. Quantize and export TFLite model
7. Generate ESPHome manifest and Wyoming config

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import yaml
import shutil
import logging
import subprocess
import tempfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import numpy as np
import scipy.io.wavfile as wavfile
import librosa
import soundfile as sf
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values to match your wake word
# ============================================================

@dataclass
class TrainingConfig:
    # --- Wake word ---
    wake_word: str = "hey computer"
    # IPA pronunciation (leave empty to auto-derive from text)
    wake_word_ipa: str = ""
    # Phonetically similar words the model should NOT trigger on
    negative_words: List[str] = field(default_factory=lambda: [
        "hey commuter", "hey computer program", "hey compiler"
    ])

    # --- Piper TTS ---
    # Piper voice model names (en_US voices recommended for English wake words)
    piper_voices: List[str] = field(default_factory=lambda: [
        "en_US-lessac-medium",
        "en_US-ryan-high",
        "en_US-amy-medium",
        "en_GB-alan-medium",
    ])
    # Number of positive samples per voice
    positive_samples_per_voice: int = 500
    # Number of negative samples per negative word per voice
    negative_samples_per_voice: int = 200

    # --- Audio ---
    sample_rate: int = 16000      # Required by microWakeWord
    audio_duration_s: float = 2.0 # Clip length in seconds

    # --- Feature extraction ---
    n_mels: int = 40
    frame_length_ms: int = 25
    frame_step_ms: int = 10
    fmin: float = 60.0
    fmax: float = 3800.0

    # --- Training ---
    batch_size: int = 256
    epochs: int = 100
    learning_rate: float = 1e-3
    validation_split: float = 0.15
    early_stopping_patience: int = 15

    # --- Model ---
    # Architecture: 'inception' or 'mixednet'
    architecture: str = "inception"
    # Detection threshold (lower = more sensitive, more false positives)
    threshold: float = 0.5
    # Sliding window size for post-processing
    sliding_window_size: int = 10

    # --- Output paths ---
    output_dir: str = "output"
    samples_dir: str = "samples"
    features_dir: str = "features"


cfg = TrainingConfig()

# Create output directories
for d in [cfg.output_dir, cfg.samples_dir, cfg.features_dir,
          f"{cfg.samples_dir}/positive", f"{cfg.samples_dir}/negative",
          f"{cfg.samples_dir}/background"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Configuration:")
for k, v in cfg.__dict__.items():
    print(f"  {k}: {v}")

## 2. Install Piper TTS & Download Voice Models

In [ ]:
# Install Piper TTS (if not already installed)
try:
    import piper
    print("Piper already installed")
except ImportError:
    print("Installing Piper TTS...")
    subprocess.run([sys.executable, "-m", "pip", "install", "piper-tts", "piper-phonemize"],
                   check=True, capture_output=True)

In [ ]:
PIPER_MODELS_DIR = Path("piper_models")
PIPER_MODELS_DIR.mkdir(exist_ok=True)

PIPER_MODEL_BASE_URL = "https://huggingface.co/rhasspy/piper-voices/resolve/main"


def download_piper_voice(voice_name: str, models_dir: Path) -> Path:
    """Download a Piper voice model if not already cached."""
    import requests

    parts = voice_name.split("-")  # e.g. en_US-lessac-medium
    lang_region = parts[0]         # en_US
    lang = lang_region.split("_")[0]  # en
    speaker = parts[1]             # lessac
    quality = parts[2]             # medium

    model_dir = models_dir / voice_name
    model_dir.mkdir(exist_ok=True)

    model_file = model_dir / f"{voice_name}.onnx"
    config_file = model_dir / f"{voice_name}.onnx.json"

    base_path = f"{lang}/{lang_region}/{speaker}/{quality}"

    for filename, dest in [
        (f"{voice_name}.onnx", model_file),
        (f"{voice_name}.onnx.json", config_file),
    ]:
        if not dest.exists():
            url = f"{PIPER_MODEL_BASE_URL}/{base_path}/{filename}"
            print(f"Downloading {url} ...")
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"  Saved {dest}")
        else:
            print(f"  Already cached: {dest}")

    return model_file


voice_model_paths = {}
for voice in cfg.piper_voices:
    try:
        path = download_piper_voice(voice, PIPER_MODELS_DIR)
        voice_model_paths[voice] = path
        print(f"✓ {voice}")
    except Exception as e:
        print(f"✗ {voice}: {e}")

print(f"\n{len(voice_model_paths)} voice(s) ready")

## 3. Generate Training Samples with Piper TTS

In [ ]:
def generate_piper_samples(
    text: str,
    model_path: Path,
    output_dir: Path,
    n_samples: int,
    sample_rate: int = 16000,
    speed_range: Tuple[float, float] = (0.85, 1.15),
) -> List[Path]:
    """
    Generate audio samples using Piper TTS with speed variation.
    Returns list of generated WAV file paths.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    generated = []

    speeds = np.linspace(speed_range[0], speed_range[1], n_samples)
    np.random.shuffle(speeds)  # randomise order

    for i, speed in enumerate(tqdm(speeds, desc=f"Generating '{text}'", leave=False)):
        out_file = output_dir / f"{i:05d}.wav"
        if out_file.exists():
            generated.append(out_file)
            continue

        try:
            cmd = [
                "piper",
                "--model", str(model_path),
                "--output_file", str(out_file),
                "--length_scale", f"{1.0 / speed:.3f}",
            ]
            result = subprocess.run(
                cmd,
                input=text.encode(),
                capture_output=True,
                timeout=30,
            )
            if result.returncode == 0 and out_file.exists():
                # Resample to target rate if needed
                audio, sr = librosa.load(str(out_file), sr=sample_rate, mono=True)
                sf.write(str(out_file), audio, sample_rate, subtype="PCM_16")
                generated.append(out_file)
            else:
                logger.warning(f"Piper failed for sample {i}: {result.stderr.decode()[:200]}")
        except subprocess.TimeoutExpired:
            logger.warning(f"Piper timed out for sample {i}")
        except Exception as e:
            logger.warning(f"Error generating sample {i}: {e}")

    return generated


# Generate positive samples (wake word)
all_positive_files = []
for voice_name, model_path in voice_model_paths.items():
    pos_dir = Path(cfg.samples_dir) / "positive" / voice_name
    files = generate_piper_samples(
        text=cfg.wake_word,
        model_path=model_path,
        output_dir=pos_dir,
        n_samples=cfg.positive_samples_per_voice,
        sample_rate=cfg.sample_rate,
    )
    all_positive_files.extend(files)
    print(f"  {voice_name}: {len(files)} positive samples")

print(f"\nTotal positive samples: {len(all_positive_files)}")

In [ ]:
# Generate negative samples (confusable words)
all_negative_files = []
for voice_name, model_path in voice_model_paths.items():
    for neg_word in cfg.negative_words:
        neg_dir = Path(cfg.samples_dir) / "negative" / voice_name / neg_word.replace(" ", "_")
        files = generate_piper_samples(
            text=neg_word,
            model_path=model_path,
            output_dir=neg_dir,
            n_samples=cfg.negative_samples_per_voice,
            sample_rate=cfg.sample_rate,
        )
        all_negative_files.extend(files)

print(f"Total negative samples: {len(all_negative_files)}")

## 4. Download Background Noise Dataset

In [ ]:
import requests
import tarfile

BG_NOISE_DIR = Path(cfg.samples_dir) / "background"
BG_NOISE_DIR.mkdir(exist_ok=True)

# MIT RIRS & Noises dataset subset (freely available)
MUSAN_NOISE_URL = "https://openslr.elda.org/resources/17/musan.tar.gz"

musan_tar = BG_NOISE_DIR / "musan.tar.gz"
musan_dir = BG_NOISE_DIR / "musan"

if not musan_dir.exists():
    print("Downloading MUSAN noise dataset (~1 GB)...")
    print("(This may take several minutes)")
    # Stream download with progress
    r = requests.get(MUSAN_NOISE_URL, stream=True, timeout=600)
    total = int(r.headers.get('content-length', 0))
    with open(musan_tar, 'wb') as f, tqdm(
        desc="MUSAN", total=total, unit='B', unit_scale=True
    ) as bar:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk)
            bar.update(len(chunk))
    print("Extracting...")
    with tarfile.open(musan_tar) as tar:
        # Validate member paths to prevent TarSlip path traversal
        for member in tar.getmembers():
            member_path = Path(member.name).resolve()
            try:
                member_path.relative_to(BG_NOISE_DIR.resolve())
            except ValueError:
                raise RuntimeError(f"Unsafe path in archive: {member.name}")
        tar.extractall(BG_NOISE_DIR)
    musan_tar.unlink()  # free space
    print("MUSAN ready")
else:
    print("MUSAN already downloaded")

# Collect all noise files
noise_files = list(musan_dir.rglob("*.wav")) if musan_dir.exists() else []
print(f"Noise files available: {len(noise_files)}")

## 5. Audio Augmentation

In [ ]:
from audiomentations import (
    Compose, AddBackgroundNoise, RoomSimulator,
    TimeStretch, PitchShift, Gain, AddGaussianNoise,
    HighPassFilter, LowPassFilter, ApplyImpulseResponse,
)


def build_augmentation_pipeline(
    noise_files: List[Path],
    sample_rate: int,
) -> Compose:
    """Build an audiomentations augmentation pipeline."""
    transforms = [
        Gain(min_gain_db=-6, max_gain_db=6, p=0.8),
        AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
        TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
        PitchShift(min_semitones=-2, max_semitones=2, p=0.4),
        HighPassFilter(min_cutoff_freq=80, max_cutoff_freq=300, p=0.3),
        LowPassFilter(min_cutoff_freq=4000, max_cutoff_freq=7500, p=0.3),
        RoomSimulator(p=0.5),
    ]

    if noise_files:
        transforms.append(
            AddBackgroundNoise(
                sounds_path=[str(p) for p in noise_files[:500]],
                min_snr_db=5,
                max_snr_db=30,
                p=0.7,
            )
        )

    return Compose(transforms)


augment = build_augmentation_pipeline(noise_files, cfg.sample_rate)
print("Augmentation pipeline ready")

## 6. Feature Extraction — Mel Spectrograms

In [ ]:
FEATURES_DIR = Path(cfg.features_dir)
FEATURES_DIR.mkdir(exist_ok=True)

FRAME_LENGTH = int(cfg.sample_rate * cfg.frame_length_ms / 1000)
FRAME_STEP = int(cfg.sample_rate * cfg.frame_step_ms / 1000)
CLIP_SAMPLES = int(cfg.sample_rate * cfg.audio_duration_s)
N_FRAMES = 1 + (CLIP_SAMPLES - FRAME_LENGTH) // FRAME_STEP

print(f"Frame length: {FRAME_LENGTH} samples ({cfg.frame_length_ms} ms)")
print(f"Frame step:   {FRAME_STEP} samples ({cfg.frame_step_ms} ms)")
print(f"Clip length:  {CLIP_SAMPLES} samples ({cfg.audio_duration_s} s)")
print(f"Feature shape per clip: ({N_FRAMES}, {cfg.n_mels}, 1)")


def load_and_pad(path: Path, target_samples: int, sample_rate: int) -> np.ndarray:
    """Load a WAV file and pad/trim to target_samples."""
    audio, _ = librosa.load(str(path), sr=sample_rate, mono=True)
    if len(audio) < target_samples:
        audio = np.pad(audio, (0, target_samples - len(audio)))
    else:
        audio = audio[:target_samples]
    return audio.astype(np.float32)


def extract_features(audio: np.ndarray, sample_rate: int, cfg: TrainingConfig) -> np.ndarray:
    """Extract log mel-spectrogram features from audio."""
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sample_rate,
        n_mels=cfg.n_mels,
        n_fft=FRAME_LENGTH,
        hop_length=FRAME_STEP,
        fmin=cfg.fmin,
        fmax=cfg.fmax,
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    # Normalize to [0, 1]
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
    # Shape: (n_mels, n_frames) -> (n_frames, n_mels, 1)
    return log_mel.T[..., np.newaxis]  # (n_frames, n_mels, 1)


def process_files(
    files: List[Path],
    label: int,
    cfg: TrainingConfig,
    augment_fn=None,
    augment_copies: int = 2,
) -> Tuple[np.ndarray, np.ndarray]:
    """Process a list of audio files into feature arrays."""
    X_list, y_list = [], []

    for path in tqdm(files, desc=f"Processing label={label}"):
        try:
            audio = load_and_pad(path, CLIP_SAMPLES, cfg.sample_rate)
        except Exception as e:
            logger.warning(f"Could not load {path}: {e}")
            continue

        copies = [audio]
        if augment_fn is not None:
            for _ in range(augment_copies):
                try:
                    copies.append(augment_fn(audio, sample_rate=cfg.sample_rate))
                except Exception:
                    pass

        for a in copies:
            feat = extract_features(a, cfg.sample_rate, cfg)
            X_list.append(feat)
            y_list.append(label)

    if not X_list:
        return np.zeros((0, N_FRAMES, cfg.n_mels, 1), dtype=np.float32), np.empty((0,), dtype=np.int32)

    return np.stack(X_list, axis=0), np.array(y_list, dtype=np.int32)


print("Feature extraction functions defined")

In [ ]:
# Build dataset
print("Processing positive samples...")
X_pos, y_pos = process_files(
    all_positive_files, label=1, cfg=cfg, augment_fn=augment, augment_copies=3
)

print("Processing negative samples...")
X_neg, y_neg = process_files(
    all_negative_files, label=0, cfg=cfg, augment_fn=augment, augment_copies=1
)

X = np.concatenate([X_pos, X_neg], axis=0)
y = np.concatenate([y_pos, y_neg], axis=0)

# Shuffle
idx = np.random.permutation(len(X))
X, y = X[idx], y[idx]

print(f"Dataset: {X.shape[0]} samples, shape {X.shape[1:]}")
print(f"  Positive: {(y == 1).sum()}")
print(f"  Negative: {(y == 0).sum()}")

# Save to disk
np.save(str(FEATURES_DIR / "X.npy"), X)
np.save(str(FEATURES_DIR / "y.npy"), y)
print("Features saved")

In [ ]:
# Visualize a few samples
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    idx_sample = np.where(y == (i % 2))[0][i // 2]
    ax.imshow(X[idx_sample, :, :, 0].T, origin='lower', aspect='auto', cmap='magma')
    ax.set_title(f"{'Wake word' if y[idx_sample] == 1 else 'Negative'} #{i // 2}")
    ax.set_xlabel("Frames")
    ax.set_ylabel("Mel bins")
plt.tight_layout()
plt.show()

## 7. Model Architecture

In [ ]:
def build_inception_model(input_shape: Tuple[int, ...]) -> tf.keras.Model:
    """
    Lightweight Inception-style model for wake word detection.
    Designed to be quantized to INT8 for ESP32 deployment.
    """
    inputs = tf.keras.Input(shape=input_shape, name="input")

    # Initial projection
    x = tf.keras.layers.Conv2D(16, (3, 3), padding="same", use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU(max_value=6.0)(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Inception block 1
    x = _inception_block(x, filters=32)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Inception block 2
    x = _inception_block(x, filters=64)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # Classifier head
    x = tf.keras.layers.Dense(64, activation="relu6")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="output")(x)

    return tf.keras.Model(inputs, outputs, name="microwakeword_inception")


def _inception_block(x, filters: int):
    """Mini inception block with 1x1, 3x3 and 5x5 convolutions."""
    assert filters % 4 == 0, f"filters must be divisible by 4, got {filters}"
    f1, f2, f3 = filters // 4, filters // 2, filters // 4

    branch1 = tf.keras.layers.Conv2D(f1, (1, 1), padding="same", use_bias=False)(x)
    branch1 = tf.keras.layers.BatchNormalization()(branch1)
    branch1 = tf.keras.layers.ReLU(max_value=6.0)(branch1)

    branch2 = tf.keras.layers.Conv2D(f2, (3, 3), padding="same", use_bias=False)(x)
    branch2 = tf.keras.layers.BatchNormalization()(branch2)
    branch2 = tf.keras.layers.ReLU(max_value=6.0)(branch2)

    branch3 = tf.keras.layers.Conv2D(f3, (5, 5), padding="same", use_bias=False)(x)
    branch3 = tf.keras.layers.BatchNormalization()(branch3)
    branch3 = tf.keras.layers.ReLU(max_value=6.0)(branch3)

    return tf.keras.layers.Concatenate()([branch1, branch2, branch3])


def build_mixednet_model(input_shape: Tuple[int, ...]) -> tf.keras.Model:
    """
    MixedNet-style model: depthwise separable convolutions for efficiency.
    Suitable for very constrained ESP32 flash / SRAM budgets.
    """
    inputs = tf.keras.Input(shape=input_shape, name="input")

    x = tf.keras.layers.Conv2D(16, (3, 3), padding="same", use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU(max_value=6.0)(x)

    for filters in [32, 48, 64]:
        x = tf.keras.layers.DepthwiseConv2D((3, 3), padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU(max_value=6.0)(x)
        x = tf.keras.layers.Conv2D(filters, (1, 1), padding="same", use_bias=False)(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.ReLU(max_value=6.0)(x)
        x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation="relu6")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="output")(x)

    return tf.keras.Model(inputs, outputs, name="microwakeword_mixednet")


INPUT_SHAPE = (N_FRAMES, cfg.n_mels, 1)

if cfg.architecture == "inception":
    model = build_inception_model(INPUT_SHAPE)
else:
    model = build_mixednet_model(INPUT_SHAPE)

model.summary()
print(f"\nTotal parameters: {model.count_params():,}")

## 8. Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Load features if needed
if 'X' not in vars() or 'y' not in vars():
    X = np.load(str(FEATURES_DIR / "X.npy"))
    y = np.load(str(FEATURES_DIR / "y.npy"))

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=cfg.validation_split,
    stratify=y,
    random_state=42,
)

# Compute class weights to handle imbalance
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train,
)
class_weight = dict(enumerate(class_weights_arr))
print(f"Class weights: {class_weight}")
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

CKPT_PATH = str(Path(cfg.output_dir) / "best_model.keras")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        patience=cfg.early_stopping_patience,
        mode="max",
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CKPT_PATH,
        monitor="val_auc",
        save_best_only=True,
        mode="max",
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc",
        factor=0.5,
        patience=7,
        mode="max",
        min_lr=1e-6,
    ),
    tf.keras.callbacks.TensorBoard(
        log_dir=str(Path(cfg.output_dir) / "logs"),
        histogram_freq=1,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=cfg.batch_size,
    epochs=cfg.epochs,
    class_weight=class_weight,
    callbacks=callbacks,
)

print("Training complete")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, title in zip(
    axes,
    ["loss", "auc", "acc"],
    ["Loss", "AUC", "Accuracy"],
):
    ax.plot(history.history[metric], label="Train")
    ax.plot(history.history[f"val_{metric}"], label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "training_curves.png"), dpi=150)
plt.show()

## 9. Quantization-Aware Training (QAT) & INT8 Export

In [ ]:
# Quantization-Aware Training (QAT) fine-tuning
print("Applying Quantization-Aware Training...")

quantize_model = tfmot.quantization.keras.quantize_model
q_aware_model = quantize_model(model)

q_aware_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate / 10),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)

q_aware_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=cfg.batch_size,
    epochs=10,
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True
        )
    ],
)

print("QAT fine-tuning complete")

In [ ]:
def convert_to_tflite_int8(
    model: tf.keras.Model,
    representative_data: np.ndarray,
    output_path: Path,
) -> None:
    """Convert a Keras model to INT8 TFLite with full integer quantization."""

    def representative_dataset():
        for i in range(min(200, len(representative_data))):
            yield [representative_data[i : i + 1].astype(np.float32)]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    tflite_model = converter.convert()
    output_path.write_bytes(tflite_model)
    size_kb = len(tflite_model) / 1024
    print(f"TFLite model saved: {output_path} ({size_kb:.1f} KB)")


TFLITE_PATH = Path(cfg.output_dir) / "model.tflite"
convert_to_tflite_int8(q_aware_model, X_val, TFLITE_PATH)

# Also save a float32 TFLite for reference
float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_model = float_converter.convert()
float_path = Path(cfg.output_dir) / "model_float32.tflite"
float_path.write_bytes(float_model)
print(f"Float32 TFLite saved: {float_path} ({len(float_model)/1024:.1f} KB)")

## 10. Generate ESPHome Manifest

In [ ]:
def generate_esphome_manifest(
    wake_word: str,
    tflite_path: Path,
    cfg: TrainingConfig,
    output_dir: Path,
) -> Path:
    """
    Generate the ESPHome micro_wake_word manifest JSON.
    Compatible with ESPHome's micro_wake_word component.
    """
    manifest = {
        "type": "micro_wake_word_model",
        "wake_word": wake_word,
        "version": 2,
        "micro": {
            "model": tflite_path.name,
            "probability_cutoff": cfg.threshold,
            "sliding_window_average_size": cfg.sliding_window_size,
        },
        "trained_languages": ["en"],
        "author": "MicroWakeWordV2Trainer",
        "architecture": cfg.architecture,
        "feature_extraction": {
            "sample_rate": cfg.sample_rate,
            "n_mels": cfg.n_mels,
            "frame_length_ms": cfg.frame_length_ms,
            "frame_step_ms": cfg.frame_step_ms,
            "fmin": cfg.fmin,
            "fmax": cfg.fmax,
        },
    }

    manifest_path = output_dir / "manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Manifest saved: {manifest_path}")
    print(json.dumps(manifest, indent=2))
    return manifest_path


manifest_path = generate_esphome_manifest(
    wake_word=cfg.wake_word,
    tflite_path=TFLITE_PATH,
    cfg=cfg,
    output_dir=Path(cfg.output_dir),
)

In [ ]:
# Print ESPHome YAML snippet
esphome_yaml = f"""
# ESPHome configuration snippet for {cfg.wake_word}
micro_wake_word:
  on_wake_word_detected:
    - logger.log: \"{cfg.wake_word} detected!\"
    # Add your actions here (e.g., start voice assistant pipeline)
  models:
    - model: http://YOUR_SERVER/{cfg.wake_word.replace(' ', '_')}/manifest.json
"""
print(esphome_yaml)

with open(Path(cfg.output_dir) / "esphome_snippet.yaml", "w") as f:
    f.write(esphome_yaml)

## 11. Wyoming Protocol Integration

In [ ]:
WYOMING_CONFIG_PATH = Path(cfg.output_dir) / "wyoming_config.yaml"

wyoming_config = {
    "wake_word": cfg.wake_word,
    "model_path": str(Path(cfg.output_dir) / "model_float32.tflite"),
    "threshold": cfg.threshold,
    "sliding_window_size": cfg.sliding_window_size,
    "sample_rate": cfg.sample_rate,
    "audio_duration_s": cfg.audio_duration_s,
    "n_mels": cfg.n_mels,
    "frame_length_ms": cfg.frame_length_ms,
    "frame_step_ms": cfg.frame_step_ms,
    "wyoming": {
        "host": "0.0.0.0",
        "port": 10400,
        "name": f"microwakeword_{cfg.wake_word.replace(' ', '_')}",
    },
}

with open(WYOMING_CONFIG_PATH, "w") as f:
    yaml.dump(wyoming_config, f, default_flow_style=False)

print(f"Wyoming config saved: {WYOMING_CONFIG_PATH}")
print(yaml.dump(wyoming_config, default_flow_style=False))

In [ ]:
WYOMING_SERVER_CODE = '''
#!/usr/bin/env python3
"""
Wyoming protocol server for microWakeWord model.
Streams audio from Home Assistant and detects the wake word.

Usage:
    python wyoming_server.py --config output/wyoming_config.yaml
"""
import argparse
import asyncio
import logging
from pathlib import Path
from collections import deque

import numpy as np
import yaml
import librosa
import tensorflow as tf
from wyoming.server import AsyncTcpServer
from wyoming.wake import Detection, NotDetected
from wyoming.audio import AudioChunk, AudioStart, AudioStop
from wyoming.event import Event
from wyoming.info import (
    Describe, Info, WakeModel, WakeProgram, Attribution
)

logger = logging.getLogger(__name__)


class WakeWordDetector:
    """Sliding-window wake word detector using TFLite."""

    def __init__(self, config: dict):
        self.cfg = config
        self.sample_rate = config["sample_rate"]
        self.threshold = config["threshold"]
        self.window = config["sliding_window_size"]
        self.frame_step = int(self.sample_rate * config["frame_step_ms"] / 1000)
        self.frame_len = int(self.sample_rate * config["frame_length_ms"] / 1000)
        self.n_mels = config["n_mels"]

        # Load TFLite interpreter
        self.interpreter = tf.lite.Interpreter(model_path=config["model_path"])
        self.interpreter.allocate_tensors()
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()

        # Audio buffer and score history
        clip_samples = int(self.sample_rate * config.get("audio_duration_s", 2.0))
        self._audio_buf = np.zeros(clip_samples, dtype=np.float32)
        self._scores: deque = deque(maxlen=self.window)

    def process_chunk(self, pcm_bytes: bytes) -> float:
        """Process a raw PCM audio chunk and return detection probability."""
        chunk = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32) / 32768.0
        self._audio_buf = np.roll(self._audio_buf, -len(chunk))
        self._audio_buf[-len(chunk):] = chunk

        # Feature extraction
        mel = librosa.feature.melspectrogram(
            y=self._audio_buf,
            sr=self.sample_rate,
            n_mels=self.n_mels,
            n_fft=self.frame_len,
            hop_length=self.frame_step,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        feat = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-8)
        feat = feat.T[np.newaxis, ..., np.newaxis].astype(np.float32)

        self.interpreter.set_tensor(self.input_details[0]["index"], feat)
        self.interpreter.invoke()
        score = float(self.interpreter.get_tensor(self.output_details[0]["index"])[0, 0])

        self._scores.append(score)
        return float(np.mean(self._scores))

    @property
    def detected(self) -> bool:
        if len(self._scores) < self.window:
            return False
        return float(np.mean(self._scores)) >= self.threshold

    def reset(self):
        self._scores.clear()


class WyomingWakeWordHandler:
    """Handles Wyoming protocol events."""

    def __init__(self, detector: WakeWordDetector, wake_word: str):
        self.detector = detector
        self.wake_word = wake_word
        self._streaming = False

    async def handle_event(self, event: Event, writer) -> None:
        if Describe.is_type(event.type):
            info = Info(
                wake=[
                    WakeProgram(
                        name="microwakeword",
                        description="MicroWakeWord V2 trained model",
                        attribution=Attribution(
                            name="MicroWakeWordV2Trainer",
                            url="https://github.com/JohnnyPrimus/MicroWakeWordV2Trainer",
                        ),
                        installed=True,
                        models=[
                            WakeModel(
                                name=self.wake_word,
                                description=f"Custom wake word: {self.wake_word}",
                                phrase=self.wake_word,
                                languages=["en"],
                                installed=True,
                                attribution=Attribution(
                                    name="MicroWakeWordV2Trainer",
                                    url="https://github.com/JohnnyPrimus/MicroWakeWordV2Trainer",
                                ),
                            )
                        ],
                    )
                ]
            )
            await writer.write_event(info.event())

        elif AudioStart.is_type(event.type):
            self._streaming = True
            self.detector.reset()

        elif AudioChunk.is_type(event.type) and self._streaming:
            chunk = AudioChunk.from_event(event)
            self.detector.process_chunk(chunk.audio)
            if self.detector.detected:
                logger.info("Wake word detected!")
                await writer.write_event(
                    Detection(name=self.wake_word, timestamp=None).event()
                )
                self.detector.reset()
                self._streaming = False

        elif AudioStop.is_type(event.type):
            self._streaming = False
            await writer.write_event(NotDetected().event())


async def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--debug", action="store_true")
    args = parser.parse_args()

    logging.basicConfig(level=logging.DEBUG if args.debug else logging.INFO)

    with open(args.config) as f:
        config = yaml.safe_load(f)

    detector = WakeWordDetector(config)
    wyoming_cfg = config["wyoming"]

    logger.info(f"Starting Wyoming server on {wyoming_cfg[\'host\']}:{wyoming_cfg[\'port\']}")

    server = AsyncTcpServer(wyoming_cfg["host"], wyoming_cfg["port"])
    handler = WyomingWakeWordHandler(detector, config["wake_word"])

    async def handle_client(reader, writer):
        while True:
            try:
                event = await reader.read_event()
                if event is None:
                    break
                await handler.handle_event(event, writer)
            except Exception as e:
                logger.error(f"Client error: {e}")
                break

    await server.run(handle_client)


if __name__ == "__main__":
    asyncio.run(main())
'''

server_path = Path("wyoming_server.py")
server_path.write_text(WYOMING_SERVER_CODE)
print(f"Wyoming server script saved: {server_path}")
print("\nTo run the Wyoming server:")
print(f"  python wyoming_server.py --config {WYOMING_CONFIG_PATH}")

## 12. Model Evaluation

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
)

# Evaluate float model on validation set
y_pred_prob = model.predict(X_val, batch_size=256).flatten()
y_pred = (y_pred_prob >= cfg.threshold).astype(int)

print("=== Classification Report ===")
print(classification_report(y_val, y_pred, target_names=["Negative", "Wake word"]))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Negative", "Wake word"])
axes[0].set_yticklabels(["Negative", "Wake word"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14)
plt.colorbar(im, ax=axes[0])

# ROC curve
fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
auc = roc_auc_score(y_val, y_pred_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(Path(cfg.output_dir) / "evaluation.png"), dpi=150)
plt.show()

print(f"\nAUC: {auc:.4f}")
tn, fp, fn, tp = cm.ravel()
print(f"False Positive Rate: {fp / (fp + tn):.4f}")
print(f"False Negative Rate: {fn / (fn + tp):.4f}")

In [ ]:
# Threshold tuning — find optimal threshold for target FPR
TARGET_FPR = 0.01  # 1% false positive rate target

fpr, tpr, thresholds = roc_curve(y_val, y_pred_prob)
valid_idx = np.where(fpr <= TARGET_FPR)[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(tpr[valid_idx])]
    recommended_threshold = thresholds[best_idx]
    print(f"At FPR <= {TARGET_FPR*100:.1f}%:")
    print(f"  Recommended threshold: {recommended_threshold:.4f}")
    print(f"  TPR (recall): {tpr[best_idx]:.4f}")
    print(f"  FPR: {fpr[best_idx]:.4f}")
    print(f"\nUpdate your manifest with: 'probability_cutoff': {recommended_threshold:.4f}")
else:
    print(f"No threshold achieves FPR <= {TARGET_FPR*100:.1f}%. Consider more negative samples.")

## 13. Summary & Output Files

In [ ]:
output_files = list(Path(cfg.output_dir).rglob("*"))

print("=" * 60)
print(f"  MicroWakeWord V2 Training Complete")
print("=" * 60)
print(f"  Wake word:   {cfg.wake_word}")
print(f"  Architecture: {cfg.architecture}")
print(f"  AUC:         {auc:.4f}")
print()
print("Output files:")
for f in sorted(output_files):
    if f.is_file():
        size = f.stat().st_size
        unit = "KB" if size > 1024 else "B"
        size_str = f"{size/1024:.1f} {unit}" if size > 1024 else f"{size} B"
        print(f"  {str(f.relative_to(cfg.output_dir)):50s} {size_str}")

print()
print("Next steps:")
print("  1. Deploy model.tflite + manifest.json to your HTTP server")
print("  2. Add ESPHome snippet to your device config")
print("  3. For server-side detection: run wyoming_server.py")
print("  4. Test on real hardware and tune threshold if needed")